# Cinematic Short Engine — TikTok AI Director

**One notebook, reusable jobs, Google Drive outputs.** Default: English narration for US TikTok viewers, vertical 9:16. This notebook checks GPU/model availability, installs compatible dependencies, configures OmniRoute API for writing, and runs research → script → storyboard → image generation/stock fallback → narration → subtitles → clips → FFmpeg render → QA.

Important: real neural image-to-video and image generation require compatible models, downloads, and hardware. The notebook records every fallback; FFmpeg pan/zoom is never labelled neural video.

In [ ]:
# 1. Mount Drive and load the repository. Tokens are read only from Colab Secrets (never printed or committed).
from pathlib import Path
import os, sys, subprocess, shutil, json, time, base64, urllib.request
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO_URL = 'https://github.com/havd58863-ui/cinematic-short-engine.git'
PROJECT_ROOT = Path('/content/cinematic-short-engine')
DRIVE_ROOT = Path('/content/drive/MyDrive/TikTokAIDirector')
OUTPUT_ROOT = DRIVE_ROOT / 'jobs'
MODEL_CACHE = DRIVE_ROOT / 'model_cache'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(MODEL_CACHE / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(MODEL_CACHE / 'huggingface' / 'hub')
os.environ['TIKTOK_OUTPUT_ROOT'] = str(OUTPUT_ROOT)
os.environ['TIKTOK_MODEL_CACHE'] = str(MODEL_CACHE)

# Optional private-repository access: create a Colab Secret named GITHUB_TOKEN only if needed.
# For a public repo, no GitHub token is required. The token is not embedded in the remote URL.
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    token = None
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if token:
        header = 'AUTHORIZATION: basic ' + base64.b64encode(('x-access-token:' + token).encode()).decode()
        cmd = ['git', '-c', 'http.extraheader=' + header, 'clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)]
    else:
        cmd = ['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print((proc.stderr or proc.stdout)[-2500:])
        raise RuntimeError('Could not clone the repository. If it is private, add a read-enabled GITHUB_TOKEN in Colab Secrets, or upload/extract the project ZIP to Drive and set PROJECT_ROOT to that folder.')
else:
    print('Project already present; reusing it.')
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError(f'No pyproject.toml found in {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('Outputs:', OUTPUT_ROOT)
print('Model cache:', MODEL_CACHE)

## 2. Install dependencies safely

Colab's current PyTorch/CUDA build is preserved. FFmpeg and espeak-ng are installed only if missing. Model weights are downloaded to Drive on first use.

In [ ]:
# Install ffmpeg and espeak-ng only if they are not already available.
if not shutil.which('ffmpeg') or not shutil.which('ffprobe') or not shutil.which('espeak-ng'):
    subprocess.run(['bash', '-lc', 'apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', str(PROJECT_ROOT), '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
# Kinocut is optional. The FFmpeg pipeline remains authoritative if installation fails.
try:
    if not shutil.which('kino'):
        kinocut_install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kinocut==1.16.1'], capture_output=True, text=True, timeout=600)
        if kinocut_install.returncode == 0:
            print('Optional Kinocut installed; CLI:', shutil.which('kino'))
        else:
            print('Optional Kinocut could not be installed; FFmpeg/ffprobe QA remains enabled.')
except Exception as kinocut_exc:
    print('Optional Kinocut skipped:', type(kinocut_exc).__name__, str(kinocut_exc)[:300])

print('Dependency installation completed. Torch was not pinned or forcibly replaced.')
print('FFmpeg:', shutil.which('ffmpeg'), '| ffprobe:', shutil.which('ffprobe'))

## 3. Prepare the OmniRoute scriptwriting API

The system uses OmniRoute API (`https://omniroute.online/`) for text generation and structured output when `OMNIROUTE_API_KEY` is provided in Colab Secrets or environment variables. If offline or unconfigured, it falls back gracefully to programmatic script/storyboard generation.

In [ ]:
import os
from google.colab import userdata

try:
    api_key = userdata.get('OMNIROUTE_API_KEY')
    if api_key:
        os.environ['OMNIROUTE_API_KEY'] = api_key
        print('OmniRoute API key loaded from Colab Secrets.')
    else:
        print('OMNIROUTE_API_KEY not found in Colab Secrets; will use deterministic fallbacks if offline.')
except Exception:
    if os.environ.get('OMNIROUTE_API_KEY'):
        print('OmniRoute API key detected in environment variables.')
    else:
        print('No OmniRoute API key detected; fallback script and storyboard builders remain available.')

## 4. Runtime preflight and automated tests

A test pass validates core logic and a small FFmpeg rendering path. It does not claim that large diffusion models were downloaded or GPU-tested.

In [ ]:
import torch, psutil
from generation.image_generation import environment_report
print(json.dumps(environment_report(), indent=2))
print('System RAM GB:', round(psutil.virtual_memory().total / 1024**3, 2))
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(PROJECT_ROOT / 'tests')], cwd=PROJECT_ROOT, check=True)

## 5. Choose how to create this TikTok

`MODE` is active: `auto` researches and selects a topic; `topic` researches your topic and writes a script; `script` preserves the supplied script and builds the storyboard/media; `resume` continues an existing job. `RUN_MEDIA=False` generates only research, script and storyboard. Licensed music is optional via `MUSIC_FILE`.

In [ ]:
# Change only the values below; all production stages are connected in run_job().
MODE = 'auto'                 # 'auto', 'topic', 'script', or 'resume'
TOPIC = None                  # Example: 'Why some emergency alerts report impossible weather events'
SCRIPT_TEXT = None            # Paste a full script here to preserve it as supplied
SCRIPT_FILE = None            # Or a text file path in Drive, e.g. '/content/drive/MyDrive/script.txt'
RESUME_JOB = None             # Job ID or full job directory path; required in MODE='resume'
RUN_MEDIA = True              # False = research/script/storyboard only
MUSIC_FILE = None             # Optional licensed audio path in Drive

# Optional stock-image fallbacks, OmniRoute configuration, and optional existing writer integration are read only from Colab Secrets.
for secret_name in ('PIXABAY_API_KEY', 'PEXELS_API_KEY', 'ENABLE_IP_ADAPTER', 'ENABLE_REAL_IMAGE_TO_VIDEO', 'OMNIROUTE_API_KEY', 'OMNIROUTE_MODEL', 'OMNIROUTE_BASE_URL', 'SCRIPTWRITER_COMMAND', 'ENABLE_PYTRENDS'):
    try:
        secret_value = userdata.get(secret_name)
        if secret_value and not os.environ.get(secret_name):
            os.environ[secret_name] = secret_value
    except Exception:
        pass

if SCRIPT_FILE:
    SCRIPT_TEXT = Path(SCRIPT_FILE).read_text(encoding='utf-8')
if MODE not in {'auto', 'topic', 'script', 'resume'}:
    raise ValueError("MODE must be one of: auto, topic, script, resume")
if MODE == 'auto':
    active_topic, active_script, active_resume = None, None, None
elif MODE == 'topic':
    if not TOPIC or not TOPIC.strip():
        raise ValueError("MODE='topic' requires TOPIC.")
    active_topic, active_script, active_resume = TOPIC.strip(), None, None
elif MODE == 'script':
    if not SCRIPT_TEXT or not SCRIPT_TEXT.strip():
        raise ValueError("MODE='script' requires SCRIPT_TEXT or SCRIPT_FILE.")
    active_topic, active_script, active_resume = (TOPIC.strip() if TOPIC and TOPIC.strip() else None), SCRIPT_TEXT.strip(), None
else:
    if not RESUME_JOB:
        raise ValueError("MODE='resume' requires RESUME_JOB.")
    active_topic, active_script, active_resume = None, None, RESUME_JOB

from tiktok_director.config import Settings
from tiktok_director.pipeline import run_job
settings = Settings.from_json(PROJECT_ROOT / 'config' / 'project_settings.json')
settings.output_root = OUTPUT_ROOT
settings.model_cache_root = MODEL_CACHE
settings.video_seconds = int(os.environ.get('DEFAULT_VIDEO_SECONDS', settings.video_seconds))
settings.omniroute_model = os.environ.get('OMNIROUTE_MODEL', settings.omniroute_model)
settings.enable_real_image_to_video = os.environ.get('ENABLE_REAL_IMAGE_TO_VIDEO', 'true').lower() == 'true'
settings.use_ip_adapter = os.environ.get('ENABLE_IP_ADAPTER', str(settings.use_ip_adapter).lower()).lower() == 'true'
settings.music_file = Path(MUSIC_FILE) if MUSIC_FILE else None

job_dir = None
try:
    job_dir = run_job(topic=active_topic, script=active_script, settings=settings, resume_job=active_resume, run_media=RUN_MEDIA)
    print('Job:', job_dir)
    manifest = json.loads((job_dir / 'run_manifest.json').read_text(encoding='utf-8'))
    print('Status:', manifest.get('status'))
except Exception as exc:
    print('Pipeline stopped:', type(exc).__name__, str(exc))
    # Find the latest persisted failure/success report; completed work is retained for resume.
    jobs = sorted([p for p in OUTPUT_ROOT.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    if jobs:
        job_dir = jobs[0]
        report_path = job_dir / 'quality_report.md'
        if report_path.exists(): print(report_path.read_text(encoding='utf-8'))
        print("Resume this job with MODE = 'resume' and RESUME_JOB =", job_dir.name)
    raise


## 6. Review the report and final MP4

If a final MP4 exists, play it here. The job report states whether each shot used neural image-to-video or FFmpeg animation, and lists any warning/failure.

In [ ]:
if job_dir is not None:
    print('Job directory:', job_dir)
    for candidate in ('research/research_report.json', 'script/approved_script.txt', 'script/script_analysis.json', 'storyboard/storyboard.json', 'quality_report.md', 'quality_report.json', 'run_manifest.json'):
        path = job_dir / candidate
        print(('FOUND  ' if path.exists() else 'MISSING'), candidate)
    report_path = job_dir / 'quality_report.md'
    if report_path.exists(): print('\n' + report_path.read_text(encoding='utf-8'))
    video_path = job_dir / 'final_video.mp4'
    if video_path.exists():
        from IPython.display import Video, display
        display(Video(filename=str(video_path), embed=True, width=360))
        print('Final MP4:', video_path)
        print('Drive folder:', job_dir)